# Análise da População 2010 x 2022 (IBGE)

Notebook para ler a tabela de população compatibilizada, agregar por Estado e por Município, e calcular o crescimento populacional entre 2010 e 2022.

## 1. Importar bibliotecas

In [ ]:
import pandas as pd


## 2. Ler a tabela

O arquivo tem 2 linhas de título antes do cabeçalho real (linha 3, índice 2) e algumas linhas de nota de rodapé no final, que serão removidas.

In [ ]:
caminho_arquivo = "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

df = pd.read_excel(caminho_arquivo, sheet_name="Municípios", header=2)

# Remove coluna vazia criada pela formatação do Excel
df = df.drop(columns=[c for c in df.columns if "Unnamed" in str(c)])

# Remove linhas de rodapé (notas/fonte), que não possuem UF preenchida
df = df.dropna(subset=["UF"]).reset_index(drop=True)

# Renomeia colunas para facilitar o uso
df = df.rename(columns={
    "População Município 2010\n(Sinopse)": "Pop_2010_Sinopse",
    "População 2010 (Alterações de Limites até 2022)1": "Pop_2010_Compatibilizada",
    "População Censo 2022": "Pop_2022"
})

df.head()


## 3. Tabela agregada por Estado

Somamos as populações compatibilizadas de 2010 e a população de 2022 para cada estado (UF).

In [ ]:
pop_estado = (
    df.groupby("UF")[["Pop_2010_Compatibilizada", "Pop_2022"]]
    .sum()
    .reset_index()
)

pop_estado.head()


## 4. Calcular a diferença (crescimento) e ordenar

Calculamos a diferença absoluta entre 2022 e 2010 (compatibilizada) e ordenamos do estado que mais cresceu para o que menos cresceu.

In [ ]:
pop_estado["Crescimento_2010_2022"] = pop_estado["Pop_2022"] - pop_estado["Pop_2010_Compatibilizada"]

pop_estado = pop_estado.sort_values("Crescimento_2010_2022", ascending=False).reset_index(drop=True)

pop_estado


## 5. Salvar tabela de estados em CSV

In [ ]:
pop_estado.to_csv(r"populacao_por_estado.csv", sep=";", index=False)
print("Arquivo salvo: populacao_por_estado.csv")


## 6. Tabela agregada por Município

Cada município já aparece em uma única linha na base original, mas fazemos o `groupby` para garantir a agregação correta caso existam municípios duplicados (por exemplo, por desmembramento territorial).

In [ ]:
pop_municipio = (
    df.groupby(["UF", "COD. MUNIC", "NOME DO MUNICÍPIO"])[["Pop_2010_Compatibilizada", "Pop_2022"]]
    .sum()
    .reset_index()
)

pop_municipio.head()


## 7. Calcular a diferença (crescimento) e ordenar por município

In [ ]:
pop_municipio["Crescimento_2010_2022"] = pop_municipio["Pop_2022"] - pop_municipio["Pop_2010_Compatibilizada"]

pop_municipio = pop_municipio.sort_values("Crescimento_2010_2022", ascending=False).reset_index(drop=True)

pop_municipio.head(20)


## 8. Salvar tabela de municípios em CSV

In [ ]:
pop_municipio.to_csv(r"populacao_por_municipio.csv", sep=";", index=False)
print("Arquivo salvo: populacao_por_municipio.csv")
